# Practical 6: N-Grams and N-Gram Smoothing

**Mapped Course Outcome:** CO4  
**BT Level Mapping:** BT3, BT4

This practical combines n-gram generation/frequency analysis with smoothing for n-gram language models.

## Aim

To generate unigram, bigram, and trigram sequences from text, analyse their frequencies, and compare MLE, Laplace, and interpolated Kneser--Ney probability estimates for seen and unseen n-grams.

## Objectives

- Generate and count unigram, bigram, and trigram sequences.
- Interpret the context--sparsity trade-off as n increases.
- Train MLE, Laplace, and interpolated Kneser--Ney language models.
- Compare probabilities for seen and unseen context-word events.
- Explain why smoothing is needed in language modeling.

## Tools / Requirements

- Python 3.10 or above
- Library: `nltk`
- Python standard-library module: `collections`
- Jupyter Notebook, JupyterLab, VS Code, or Google Colab

If required, run `%pip install nltk` in a notebook cell and restart the kernel. This notebook uses `wordpunct_tokenize`, so it does **not** need a separate NLTK tokenizer download.

## Theory

An **n-gram** is a contiguous sequence of *n* tokens. Unigrams have one token, bigrams have two adjacent tokens, and trigrams have three. Longer n-grams carry more local context, but exact longer sequences occur less often. This spreading of observations over many possible sequences is called **data sparsity**.

For a count-based model, the maximum-likelihood estimate (MLE) is $P(w_i \mid h) = C(h, w_i) / C(h)$. Therefore, an unseen n-gram has probability zero under MLE. **Smoothing** transfers a small amount of probability mass away from observed events so unseen events can receive non-zero probability. Laplace smoothing adds one to every count. Interpolated Kneser--Ney uses discounting and lower-order continuation information, and is usually more useful for sparse n-gram models.

Start (`<s>`) and end (`</s>`) markers allow a language model to learn sentence beginnings and endings.

## Methodology

1. Prepare and lowercase a small sentence corpus.
2. Generate unigrams, bigrams, and trigrams within sentence boundaries.
3. Build frequency distributions and compare sparsity across orders.
4. Pad tokenized sentences and train trigram MLE, Laplace, and Kneser--Ney models.
5. Score seen and unseen context-word events and compare the estimates.
6. Evaluate the models on held-out text using perplexity.

In [1]:
from collections import Counter
import re

from nltk import ngrams
from nltk.tokenize import wordpunct_tokenize
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline

print('Imports completed successfully.')

Imports completed successfully.


## Dataset and tokenization

The small in-code corpus repeats selected language-modeling expressions. It is sufficient for demonstrating frequency patterns and sparse events; replace it with a larger tokenized corpus (for example, Brown, Reuters, or domain documents) for a realistic model.

In [2]:
text = '''Natural language processing helps computers understand human language.
Natural language processing uses language data to identify patterns.
Language models learn useful patterns from text data.
A language model assigns probabilities to word sequences.
Natural language data can improve a language model.'''

# Sentence splitting and wordpunct_tokenize avoid any NLTK-data download.
sentences = [sentence.strip() for sentence in re.split(r'[.!?]+', text) if sentence.strip()]

def tokenize(sentence):
    """Lowercase a sentence and retain alphabetic word tokens only."""
    return [token.lower() for token in wordpunct_tokenize(sentence) if token.isalpha()]

tokenized_sentences = [tokenize(sentence) for sentence in sentences]
tokens = [token for sentence in tokenized_sentences for token in sentence]

print('Number of sentences:', len(tokenized_sentences))
print('Number of tokens:', len(tokens))
print('Normalized tokens:', tokens)

Number of sentences: 5
Number of tokens: 41
Normalized tokens: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language', 'natural', 'language', 'processing', 'uses', 'language', 'data', 'to', 'identify', 'patterns', 'language', 'models', 'learn', 'useful', 'patterns', 'from', 'text', 'data', 'a', 'language', 'model', 'assigns', 'probabilities', 'to', 'word', 'sequences', 'natural', 'language', 'data', 'can', 'improve', 'a', 'language', 'model']


## Part A — Generate n-grams and count frequencies

N-grams are produced sentence by sentence. This is important: joining all tokens first would incorrectly create n-grams that cross sentence boundaries.

In [3]:
def corpus_ngrams(sentences, n):
    """Return all n-grams without crossing a sentence boundary."""
    return [gram for sentence in sentences for gram in ngrams(sentence, n)]

unigrams = corpus_ngrams(tokenized_sentences, 1)
bigrams = corpus_ngrams(tokenized_sentences, 2)
trigrams = corpus_ngrams(tokenized_sentences, 3)

unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

print('First 12 unigrams:', unigrams[:12])
print('First 12 bigrams: ', bigrams[:12])
print('First 12 trigrams:', trigrams[:12])

First 12 unigrams: [('natural',), ('language',), ('processing',), ('helps',), ('computers',), ('understand',), ('human',), ('language',), ('natural',), ('language',), ('processing',), ('uses',)]
First 12 bigrams:  [('natural', 'language'), ('language', 'processing'), ('processing', 'helps'), ('helps', 'computers'), ('computers', 'understand'), ('understand', 'human'), ('human', 'language'), ('natural', 'language'), ('language', 'processing'), ('processing', 'uses'), ('uses', 'language'), ('language', 'data')]
First 12 trigrams: [('natural', 'language', 'processing'), ('language', 'processing', 'helps'), ('processing', 'helps', 'computers'), ('helps', 'computers', 'understand'), ('computers', 'understand', 'human'), ('understand', 'human', 'language'), ('natural', 'language', 'processing'), ('language', 'processing', 'uses'), ('processing', 'uses', 'language'), ('uses', 'language', 'data'), ('language', 'data', 'to'), ('data', 'to', 'identify')]


In [4]:
def print_most_common(counts, title, limit=10):
    print(f'\n{title}')
    print(f"{'N-gram':<38} Frequency")
    print('-' * 50)
    for gram, count in counts.most_common(limit):
        print(f"{' '.join(gram):<38} {count}")

print_most_common(unigram_counts, 'Most frequent unigrams')
print_most_common(bigram_counts, 'Most frequent bigrams')
print_most_common(trigram_counts, 'Most frequent trigrams')


Most frequent unigrams
N-gram                                 Frequency
--------------------------------------------------
language                               8
natural                                3
data                                   3
processing                             2
to                                     2
patterns                               2
a                                      2
model                                  2
helps                                  1
computers                              1

Most frequent bigrams
N-gram                                 Frequency
--------------------------------------------------
natural language                       3
language processing                    2
language data                          2
a language                             2
language model                         2
processing helps                       1
helps computers                        1
computers understand                   1
understand huma

In [5]:
ngram_counters = {
    'Unigram': unigram_counts,
    'Bigram': bigram_counts,
    'Trigram': trigram_counts,
}

print(f"{'Order':<10} {'Observed':>10} {'Unique':>10} {'Unique ratio':>14} {'Max count':>12}")
print('-' * 62)
for name, counts in ngram_counters.items():
    observed = sum(counts.values())
    unique = len(counts)
    print(f"{name:<10} {observed:>10} {unique:>10} {unique / observed:>14.3f} {max(counts.values()):>12}")

print('\nInterpretation: a higher unique ratio and lower maximum count indicate greater sparsity.')

Order        Observed     Unique   Unique ratio    Max count
--------------------------------------------------------------
Unigram            41         25          0.610            8
Bigram             36         30          0.833            3
Trigram            31         29          0.935            2

Interpretation: a higher unique ratio and lower maximum count indicate greater sparsity.


## Boundary markers and an MLE count estimate

Boundary symbols make sentence starts/ends explicit. The following cell manually verifies one MLE bigram probability before training the NLTK models.

In [6]:
boundary_bigrams = [
    gram
    for sentence in tokenized_sentences
    for gram in ngrams(['<s>'] + sentence + ['</s>'], 2)
]
boundary_bigram_counts = Counter(boundary_bigrams)
context_counts = Counter(context for context, _ in boundary_bigrams)

context, word = '<s>', 'natural'
mle_probability = boundary_bigram_counts[(context, word)] / context_counts[context]
print(f"Count({context!r}, {word!r}) = {boundary_bigram_counts[(context, word)]}")
print(f"Count({context!r}) = {context_counts[context]}")
print(f"P({word} | {context}) = {mle_probability:.3f}")

Count('<s>', 'natural') = 3
Count('<s>') = 5
P(natural | <s>) = 0.600


## Part B — N-gram smoothing

A second, compact training corpus has related sentences. It includes familiar individual words but intentionally omits some valid word-context combinations. This lets us show the zero-probability problem fairly: the target word is known to the vocabulary, but its exact trigram context is not.

In [7]:
lm_train_sentences = [
    ['i', 'study', 'natural', 'language', 'processing'],
    ['we', 'study', 'natural', 'language', 'processing'],
    ['i', 'study', 'language', 'models'],
    ['we', 'study', 'language', 'models'],
    ['language', 'models', 'predict', 'words'],
    ['natural', 'language', 'data', 'predict', 'text'],
]

for sentence in lm_train_sentences:
    print(' '.join(sentence))

i study natural language processing
we study natural language processing
i study language models
we study language models
language models predict words
natural language data predict text


In [8]:
ORDER = 3

def fit_language_model(model_class, sentences, order=ORDER, **kwargs):
    """Fit one padded n-gram model with a fresh training-data generator."""
    train_data, vocabulary = padded_everygram_pipeline(order, sentences)
    model = model_class(order, **kwargs)
    model.fit(train_data, vocabulary)
    return model

mle_model = fit_language_model(MLE, lm_train_sentences)
laplace_model = fit_language_model(Laplace, lm_train_sentences)
kneser_ney_model = fit_language_model(KneserNeyInterpolated, lm_train_sentences, discount=0.1)

print('Models trained:', ', '.join(['MLE', 'Laplace', 'Interpolated Kneser-Ney']))
print('Vocabulary size (including padding/OOV symbols):', len(mle_model.vocab))

Models trained: MLE, Laplace, Interpolated Kneser-Ney
Vocabulary size (including padding/OOV symbols): 14


### Seen versus unseen events

`processing | natural language` was seen in training. `models | natural language` was not seen in that trigram context, even though `models` occurs elsewhere in the corpus. MLE should therefore assign zero to the second event, while the two smoothing methods should assign it non-zero probability.

In [9]:
models = {
    'MLE': mle_model,
    'Laplace': laplace_model,
    'Kneser-Ney': kneser_ney_model,
}
events = [
    ('Seen', 'processing', ('natural', 'language')),
    ('Unseen', 'models', ('natural', 'language')),
    ('Unseen', 'words', ('natural', 'language')),
]

raw_trigram_counts = Counter(
    gram for sentence in lm_train_sentences for gram in ngrams(sentence, ORDER)
)
assert raw_trigram_counts[('natural', 'language', 'processing')] > 0
assert raw_trigram_counts[('natural', 'language', 'models')] == 0

print(f"{'Event':<8} {'Word | context':<38} {'MLE':>12} {'Laplace':>12} {'Kneser-Ney':>14}")
print('-' * 92)
for label, word, context in events:
    scores = [model.score(word, context) for model in models.values()]
    event_name = f"{word} | {' '.join(context)}"
    print(f"{label:<8} {event_name:<38} {scores[0]:>12.6f} {scores[1]:>12.6f} {scores[2]:>14.6f}")

Event    Word | context                                  MLE      Laplace     Kneser-Ney
--------------------------------------------------------------------------------------------
Seen     processing | natural language              0.666667     0.176471       0.648561
Unseen   models | natural language                  0.000000     0.058824       0.031894
Unseen   words | natural language                   0.000000     0.058824       0.000227


## Held-out evaluation with perplexity

Perplexity measures how surprised a model is by a sequence; lower values are better when models are evaluated on the same test data. MLE can have infinite perplexity when a test sequence contains an unseen event because its probability is zero. The test sentence is not used to fit the models.

In [10]:
held_out_sentence = [['we', 'study', 'natural', 'language', 'models']]
test_data, _ = padded_everygram_pipeline(ORDER, held_out_sentence)
test_ngrams = list(next(test_data))  # Materialize once so every model sees identical data.

print('Held-out sentence:', ' '.join(held_out_sentence[0]))
for name, model in models.items():
    value = model.perplexity(test_ngrams)
    display_value = 'infinite (contains a zero-probability event)' if value == float('inf') else f'{value:.3f}'
    print(f'{name:<12}: {display_value}')

Held-out sentence: we study natural language models
MLE         : infinite (contains a zero-probability event)
Laplace     : 6.863
Kneser-Ney  : 4.632


## Results and interpretation

- Repeated expressions such as `natural language` receive higher counts.
- From unigrams to trigrams, the unique-sequence ratio rises and individual counts become smaller: this is the context--sparsity trade-off.
- MLE gives probability zero to unseen but vocabulary-known n-grams.
- Laplace and Kneser--Ney assign non-zero probability to those events by redistributing probability mass. Laplace is simple but can over-smooth; Kneser--Ney also uses lower-order continuation evidence and is typically better for sparse n-gram language models.
- A test sentence with an unseen MLE event can produce infinite perplexity, whereas smoothed models remain usable.

## Outcomes

After completing this practical, you can generate n-grams, interpret n-gram frequency distributions, explain the context--sparsity trade-off, and compare how MLE and smoothing methods treat seen and unseen language events.

## Self-assessment answers

1. An n-gram is a contiguous sequence of *n* tokens (words or characters).
2. A unigram has one token, a bigram has two adjacent tokens, and a trigram has three adjacent tokens.
3. Longer n-grams include more previous tokens, so they provide more specific local context.
4. Data sparsity means that many possible sequences have very low counts or are absent from the training data.
5. The Markov assumption estimates a word from only a limited preceding context rather than the whole history.
6. Sentence boundary symbols let the model learn how sentences start and end.
7. An MLE n-gram probability is its count divided by the count of its $(n-1)$-gram context.
8. Character n-grams are used in spelling correction and language identification.
9. An unseen n-gram with zero probability makes the probability of any sequence containing it zero; smoothing avoids this.
10. Increasing n expands the number of possible sequences, increasing memory and computational cost.
11. MLE assigns zero probability because the unseen event has count zero.
12. Probability-mass redistribution reserves some probability for unseen events by reducing the probability of observed events.
13. Laplace smoothing adds one to each possible event count before normalizing.
14. Add-one smoothing can overestimate rare/unseen events, especially with a large vocabulary.
15. Discounting reduces observed counts slightly to free probability mass for unseen events.
16. Kneser--Ney is effective because it uses discounted counts and continuation statistics from lower orders.
17. Backoff uses a lower-order model only when necessary; interpolation combines higher- and lower-order estimates for every prediction.
18. Smoothing slightly lowers probabilities of frequent observed n-grams to allocate mass to unseen events.
19. Perplexity measures a language model's predictive uncertainty on held-out text; lower is better for comparable models/test data.
20. Evaluation on unseen test data measures generalization rather than memorization of the training corpus.

## Resources

- [NLTK language-model API](https://www.nltk.org/api/nltk.lm.html)
- [Speech and Language Processing (Jurafsky & Martin)](https://web.stanford.edu/~jurafsky/slp3/)
- [NLTK Book](https://www.nltk.org/book/)
- [NPTEL n-gram language-model videos](https://www.youtube.com/results?search_query=NPTEL+N-gram+language+model+NLP)
- [NPTEL n-gram smoothing videos](https://www.youtube.com/results?search_query=NPTEL+n-gram+smoothing+Laplace+Kneser+Ney)